# Idea

Enumerate all possible combinations of symbols by vector numeric value.
For example:

Case words -- first, numbers -- last:
* For length = 1:
    * `a`, `b`, ... `z` : +26;
    * `0`, `1`, ... `9` : +10;
* For length = 2:
    * `aa`, `ab`, ... `zz` : +26 * 26;
    * `a0`, `a1`, ... `z9` : +26 * 10;
    * `00`, `01`, ... `99` : +10 * 10;
* For length = 3:
    * `aaa`, `aab`, ... `zzz` : +26 * 26 * 26;
    * `aa0`, `aa1`, ... `zz9` : +26 * 26 * 10;
    * `a00`, `a01`, ... `z99` : +26 * 10 * 10;
    * `000`, `001`, ... `999` : +10 * 10 * 10;
    * `a_a`, `a_b`, ... `z_z` : +26 * 1 * 26;
    * `0_0`, `0_1`, ... `9_9` : +10 * 1 * 10;
* For length = 4:
    * `aaaa`, `aaab`, ... `zzzz` : +26 * 26 * 26 * 26;
    * `aaa0`, `aaa1`, ... `zzz9` : +26 * 26 * 26 * 10;
    * `aa00`, `aa01`, ... `zz99` : +26 * 26 * 10 * 10;
    * `a000`, `a001`, ... `z999` : +26 * 10 * 10 * 10;
    * `0000`, `0001`, ... `9999` : +10 * 10 * 10 * 10;
    * `aa_a`, `aa_b`, ... `zz_z` : +26 * 26 * 1 * 26;
    * `a_aa`, `a_ab`, ... `z_zz` : +26 * 1 * 26 * 26;
    * `a_a0`, `a_a1`, ... `z_z9` : +26 * 1 * 26 * 10;
    * `0_00`, `0_01`, ... `9_99` : +10 * 1 * 10 * 10;
* and so on.

# Density

In [ ]:
import math

vector_lengths = [64, 128, 192, 256]

word_alphabet_size = 26 # 'a'-'z'
number_alphabet_size = 10 # '0'-'9'

def explicit_separated_ids_count(word_id_length : int, alphabet_size : int) -> int:
    count = 0
    separators_places_count = word_id_length - 1
    for separators_count in range(separators_places_count):
        letters_count = word_id_length - separators_count
        separators_combinations = math.comb(separators_places_count, separators_count)
        count += separators_combinations * (alphabet_size ** letters_count)
    return count

def ids_count(id_length : int) -> int:
    count = 0
    for words_id_length in range(id_length + 1):
        numbers_id_length = id_length - words_id_length
        count += explicit_separated_ids_count(words_id_length, word_alphabet_size) * explicit_separated_ids_count(numbers_id_length, number_alphabet_size)
    return count

for vector_length in vector_lengths:
    ids_counted = 0
    vector_max_id = 1 << vector_length
    id_length = 0

    while True:
        ids_count_per_length = ids_count(id_length)
        if ids_counted + ids_count_per_length > vector_max_id:
            break
        ids_counted += ids_count_per_length
        id_length += 1
    
    used_bits = math.ceil(math.log2(ids_counted))
    unused_bits = vector_length - used_bits
    density = used_bits / id_length

    print(f"{vector_length}-bits vector contains {id_length} symbols (guaranteed), density: {density:.3f}, unused bits: {unused_bits}, total ids count: {ids_counted:.3g}")

64-bits vector contains 14 symbols (guaranteed), density: 4.286, unused bits: 4, total ids count: 9.74e+17
128-bits vector contains 28 symbols (guaranteed), density: 4.536, unused bits: 1, total ids count: 1.07e+38
192-bits vector contains 41 symbols (guaranteed), density: 4.610, unused bits: 3, total ids count: 4.32e+56
256-bits vector contains 55 symbols (guaranteed), density: 4.636, unused bits: 1, total ids count: 4.73e+76


# Operations

All operations are expensive because any change requires full decoding and re-encoding.

- Encode ~ $O(L_{id})$ because we need to iterate through all symbols.
- Decode ~ $O(L_{id})$ because we need to iterate through all symbols.
- GetLength ~ $O(L_{id})$ because we need to decode identifier to get length.
- GetSymbol ~ $O(L_{id})$ because we need to decode identifier to access symbol by index.
- GetToken ~ $O(L_{id})$ because we need to decode identifier to access token by index.
- AppendSymbol ~ $O(L_{id})$ because we need to decode and re-encode identifier.
- AppendToken ~ $O(L_{id})$ because we need to decode and re-encode identifier.